In [4]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
import os

def rasterize_fclass_by_grid(shapefile_path, grid_path, fclass_mapping, output_dir, resolution):
    # 读取水体数据和网格
    gdf = gpd.read_file(shapefile_path)
    grid = gpd.read_file(grid_path)

    # 坐标系统一为 UTM 51N
    if not gdf.crs.is_projected:
        gdf = gdf.to_crs(epsg=32651)
    if not grid.crs.is_projected:
        grid = grid.to_crs(epsg=32651)
    if gdf.crs != grid.crs:
        grid = grid.to_crs(gdf.crs)

    # 保留指定类别的记录，并转换为数值
    gdf = gdf[gdf['fclass'].isin(fclass_mapping.keys())]
    gdf['class_val'] = gdf['fclass'].map(fclass_mapping)

    # 清理空 geometry
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notnull()]
    grid = grid[~grid.geometry.is_empty & grid.geometry.notnull()]

    os.makedirs(output_dir, exist_ok=True)

    for _, cell in grid.iterrows():
        tile_id = cell["ID"]  # 使用 ID 字段作为输出文件名
        grid_geom = cell.geometry

        # 裁剪出当前格子内的水体
        clipped = gpd.overlay(gdf, gpd.GeoDataFrame(geometry=[grid_geom], crs=gdf.crs), how="intersection")

        if clipped.empty:
            continue

        # 计算输出 raster 尺寸
        minx, miny, maxx, maxy = grid_geom.bounds
        width = int((maxx - minx) / resolution)
        height = int((maxy - miny) / resolution)
        transform = from_origin(minx, maxy, resolution, resolution)

        # 构造 shapes
        shapes = [(geom, val) for geom, val in zip(clipped.geometry, clipped['class_val'])]

        raster = rasterize(
            shapes=shapes,
            out_shape=(height, width),
            transform=transform,
            fill=255,  # Nodata 为 255
            dtype='uint8'
        )

        out_path = os.path.join(output_dir, f"water_tile_{tile_id}.tif")

        with rasterio.open(
            out_path,
            "w",
            driver="GTiff",
            height=height,
            width=width,
            count=1,
            dtype="uint8",
            crs=gdf.crs,
            transform=transform,
            nodata=255
        ) as dst:
            dst.write(raster, 1)

        print(f"✅ 输出: {out_path}")

# 示例调用
fclass_mapping = {
    'dock': 0,
    'reservoir': 1,
    'riverbank': 2,
    'water': 3,
    'wetland': 4
}

rasterize_fclass_by_grid(
    shapefile_path=r"D:\BaiduNetdiskDownload\3-shanghai-latest-free.shp\gis_osm_water_a_free_1.shp",
    grid_path=r"D:\shanghai workshop\vector\shanghai_grid_1000m_clipped.shp",
    fclass_mapping=fclass_mapping,
    output_dir=r"D:\shanghai workshop\raster\water_tiles",
    resolution=5
)


✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_47.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_48.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_49.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_180.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_181.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_182.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_183.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_184.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_313.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_314.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_315.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_316.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_317.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_440.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_441.tif
✅ 输出: D:\shanghai workshop\r